In [1]:
# Cell 1 – FIRST, before any app imports
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# Explicit path; Jupyter project root is usually /app or /code
for p in [Path("/app/.env"), Path("/code/.env"), Path.cwd() / ".env"]:
    if p.exists():
        load_dotenv(p)
        print(f"Loaded from {p}")
        break
else:
    print("No .env found")

# Sanity check
print("HUGGINGFACE_API_TOKEN set:", bool(os.getenv("HUGGINGFACE_API_TOKEN")))
print("OPENAI_API_TOKEN set:", bool(os.getenv("OPENAI_API_TOKEN")))
# override DATABASE_URL to look at local
os.environ["DATABASE_URL"] = "postgresql+psycopg2://badger:badgerpass@db:5432/badgerdb"
sys.path.append(str(Path().resolve().parent))
from app.db import engine, SessionLocal#from app.models import User, Document, VaultMembership
from uuid import UUID

import numpy as np
import pandas as pd
import json


Loaded from /app/.env
HUGGINGFACE_API_TOKEN set: True
OPENAI_API_TOKEN set: False


In [80]:
prompt_text = """
document text:
{document_text}"""

prompt_dict = {
    "ContactInfo_v1":{
        "description": "structured contact extraction",
        "text":prompt_text,
        "components":["basic_extraction"]
    },
    "basic_extraction":{
        "decription":"starting point for extractive",
        "text":"""you are a data extraction agent.  
Return a JSON object matching the schema exactly.

"""},
}

with open("prompt.config.json", "w", encoding="utf-8") as f:
    json.dump(prompt_dict, f, indent=2)
prompt_dict

{'ContactInfo_v1': {'description': 'structured contact extraction',
  'text': '\ndocument text:\n{document_text}',
  'components': ['basic_extraction']},
 'basic_extraction': {'decription': 'starting point for extractive',
  'text': 'you are a data extraction agent.  \nReturn a JSON object matching the schema exactly.\n\n'}}

In [74]:
def load_prompt(prompt_name: str, config_path: str | Path) -> type[ChatPromptTemplate]:
    raw = json.loads(Path(config_path).read_text())
    spec = raw[prompt_name]

    messages = []
    for c in prompt_dict["ContactInfo_v1"].get("components"):
        
        messages.append(("system",prompt_dict.get(c).get("text")))
    
    messages.append(("human",spec["text"]))

    return ChatPromptTemplate.from_messages(messages)

In [75]:
load_prompt('ContactInfo_v1', 'prompt.config.json')

ChatPromptTemplate(input_variables=['document_text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='you are a data extraction agent.  \nReturn a JSON object matching the schema exactly.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['document_text'], input_types={}, partial_variables={}, template='\ndocument text:\n{document_text}'), additional_kwargs={})])

In [33]:
contact_info_dict = {
  "ContactInfo": {
    "description": "Structured contact extraction",
    "fields": {
      "name": {
        "type": "str",
        "required": True,
        "description": "Full name"
      },
      "email": {
        "type": "str",
        "required": False
      },
      "category": {
        "type": "literal",
        "choices": ["lead", "customer", "partner"],
        "required": True
      },
      "notes": {
        "type": "list[str]",
        "required": False
      }
    }
  }
}

In [34]:
import json

with open("model.config.json", "w", encoding="utf-8") as f:
    json.dump(contact_info_dict, f, indent=2)

In [37]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Literal, Optional
from pydantic import BaseModel, Field, create_model, ConfigDict

TYPE_MAP = {
    "str": str,
    "int": int,
    "float": float,
    "bool": bool,
    "list[str]": list[str]
    #"any": Any,
}

def build_field(spec: dict):
    t = spec["type"]

    if t == "literal":
        choices = tuple(spec["choices"])
        py_type = Literal[choices]  # works at runtime
    else:
        py_type = TYPE_MAP[t]

    required = spec.get("required", False)
    default = ... if required else None

    field = Field(
        default,
        description=spec.get("description"),
        examples=spec.get("examples"),
    )
    return py_type, field

def load_model(model_name: str, config_path: str | Path) -> type[BaseModel]:
    raw = json.loads(Path(config_path).read_text())
    spec = raw[model_name]

    fields = {
        name: build_field(field_spec)
        for name, field_spec in spec["fields"].items()
    }

    model = create_model(
        model_name,
        __config__=ConfigDict(extra="forbid"),
        __doc__=spec.get("description"),
        **fields,
    )
    return model

In [38]:
ContactInfo = load_model("ContactInfo", "model.config.json")

#data = llm_extract(prompt, schema=ContactInfo)  # framework-specific
#obj = ContactInfo.model_validate(data)

In [39]:
ContactInfo.model_fields

{'name': FieldInfo(annotation=str, required=True, description='Full name'),
 'email': FieldInfo(annotation=str, required=False, default=None),
 'category': FieldInfo(annotation=Literal['lead', 'customer', 'partner'], required=True),
 'notes': FieldInfo(annotation=list[str], required=False, default=None)}

In [81]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
you are a data extraction agent.  
Return a JSON object matching the schema exactly.

document text:
{document_text}
""")

prompt = load_prompt('ContactInfo_v1', 'prompt.config.json')

def build_model(model_name: str = "gpt-4.1-nano", response_structure=ContactInfo):
    llm = ChatOpenAI(model=model_name, temperature=0)
    return llm.with_structured_output(response_structure, method="json_schema")

model = build_model()
extract_chain = prompt | model

In [82]:
data = """Hi, I'm John Cena.  I'd like to have more information about your awesome invisibility product.
I want to be real clear that people don't know me... they can't see what I'm about.
Please reach out with any information that you think I will find useful.

You can reach me at john.cena@youdontknowmeson.com"""

response = extract_chain.invoke({"document_text":data})
response

ContactInfo(name='John Cena', email='john.cena@youdontknowmeson.com', category='lead', notes=['Interested in invisibility product', 'Wants more information', 'Prefers discreet communication'])